# Train your own Sprout walking policy (free GPU)

This notebook trains a neural-network walking policy for the **Sprout Challenge** with reinforcement learning, the
same way the `examples/ppo_baseline` entry was trained, then turns it into an entry you can grade and submit.

**What happens:**
1. Install the GPU simulator (MuJoCo Warp) and the trainer (Brax PPO).
2. Thousands of simulated Sprouts practise walking at the same time on 8 tracks: the practice track plus
   generated ones, from easy to hard.
3. The trained network is saved as plain numpy arrays (`policy.npz`), which the grader can run on a CPU.
4. Your entry is graded on the practice track exactly like the judges do, and you get a video of it.

**Before you start:** turn on a GPU.
- **Colab:** Runtime → Change runtime type → **T4 GPU**.
- **Kaggle:** Settings → Accelerator → **GPU T4 x2** (one is used), and turn **Internet on** (needs a verified
  phone number).

**How long:** the install takes about 5 minutes. Training to a walking policy takes about 30 minutes on a desktop
GPU and longer on a free T4 (expect 1.5 to 3 hours). Free sessions can disconnect, so the trainer saves checkpoints
and can resume.

The reward and the training setup are in [`training/sprout_walk.py`](https://github.com/shantanu-builds-robots/sprout-challenge/blob/main/training/sprout_walk.py).
Changing them is the point: the baseline walks around every barrier and only reaches about 0.8 m/s.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install

Downloads the challenge and installs the GPU packages, pinned to the versions this notebook was tested with. A
second, separate Python environment gets the exact packages the judges grade with, so your local score matches.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()                      # /content on Colab, /kaggle/working on Kaggle
REPO = ROOT / "sprout-challenge"
if not REPO.exists():
    !git clone -q https://github.com/shantanu-builds-robots/sprout-challenge {REPO}

# Training packages (GPU)
!pip install -q "jax[cuda12]==0.10.2" "brax==0.14.2" "playground==0.2.0" "mujoco==3.17.0" "mujoco-mjx==3.17.0" \
    "mujoco-warp==3.17.0" "warp-lang==1.19.0" pyyaml matplotlib
sys.path.insert(0, str(REPO))

# Grading environment (CPU), the same packages as the judges' container
!pip install -q uv
!uv venv -q --python 3.11 {ROOT}/grade-env
!uv pip install -q --python {ROOT}/grade-env/bin/python "{REPO}[video]"
GRADE_PY = str(ROOT / "grade-env" / "bin" / "python")
print("installed")

## 3. Build the training world

Eight 6 m wide tracks side by side. Lane 0 is the practice track (its barrier swapped for flat ground: this policy
learns to walk, and goes around barriers through the open lane); the others are generated tracks of increasing
difficulty. Each robot starts somewhere random along a random lane.

In [ ]:
import jax, numpy as np, matplotlib.pyplot as plt
from training import sprout_walk as sw

print("GPU:", jax.devices())
env = sw.WalkEnv()
H = np.asarray(env._H)
plt.figure(figsize=(14, 3))
plt.imshow(H, origin="lower", aspect="auto", cmap="terrain",
           extent=[0, H.shape[1] * 0.05, -H.shape[0] * 0.025, H.shape[0] * 0.025])
plt.colorbar(label="height [m]"); plt.xlabel("x along the track [m]"); plt.ylabel("y [m]")
plt.title("training lanes (lane 0, the practice track, is at the bottom)"); plt.show()

## 4. Choose the training settings

`TIMESTEPS` is the total number of robot steps (each 0.02 s). The baseline was exported at about 115 million. Start
smaller (say 30 million) to check everything works before a long run.

The reward weights are below: positive weights are paid for, negative ones are penalties. They start at the
baseline's values. See `reward_terms` in `training/sprout_walk.py` for what each term measures.

In [ ]:
TIMESTEPS = 120_000_000
NUM_ENVS = 8192            # robots in parallel; use 4096 if the GPU runs out of memory
SEED = 0
CHECKPOINTS = ROOT / "checkpoints"
RESUME = None              # or a folder inside CHECKPOINTS, to continue an interrupted run

from ml_collections import config_dict
cfg = sw.default_config()
cfg.course.speed = [0.4, 0.8]                       # m/s: the forward speeds it practises
for k, v in sw.REWARD_SCALES.items():               # edit any weight here, e.g. cfg.reward.scales.progress = 2.0
    cfg.reward.scales[k] = v
print(cfg.reward.scales)

## 5. Train

The plot updates 20 times during training, each time from 128 fresh robots running the current policy without
exploration noise. **Seconds before falling** should climb towards 20 and **metres walked** towards 8 to 16 (the
practised speeds are 0.4 to 0.8 m/s). The first point takes a few minutes: compiling comes first. Checkpoints are saved to `checkpoints/`. On Colab you can copy them to Google Drive, so a
disconnect doesn't lose them (see the last cell).

In [ ]:
from IPython.display import clear_output
env = sw.WalkEnv(cfg)
hist = []

def progress(step, row):
    hist.append((step, row))
    clear_output(wait=True)
    s = [h[0] / 1e6 for h in hist]
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
    ax[0].plot(s, [h[1]["reward"] for h in hist], "o-"); ax[0].set_title("reward per episode")
    ax[1].plot(s, [h[1]["seconds"] for h in hist], "o-"); ax[1].set_title("seconds before falling (max 20)")
    ax[2].plot(s, [h[1]["distance"] for h in hist], "o-"); ax[2].set_title("metres walked in an episode")
    for a in ax: a.set_xlabel("million steps")
    plt.show()
    print(f"{step:,} steps, {row['minutes']:.0f} min, {row['sps']:,.0f} steps/s")

params = sw.train(env, num_timesteps=TIMESTEPS, num_envs=NUM_ENVS, seed=SEED, checkpoint_dir=str(CHECKPOINTS),
                  restore_dir=RESUME, progress=progress)
print("training done")

## 6. Turn it into an entry

Saves the network as `my_entry/policy.npz` next to a copy of the baseline's `controller.py`, which reads the
sensors, builds the same 139 inputs and steers around barriers. Then checks the numpy copy matches the trained
network.

In [ ]:
import shutil
ENTRY = ROOT / "my_entry"
ENTRY.mkdir(exist_ok=True)
shutil.copy(REPO / "examples" / "ppo_baseline" / "controller.py", ENTRY / "controller.py")
z = sw.export(params, ENTRY / "policy.npz")
print("numpy vs trained network, largest difference:", sw.check_export(env, params, z))   # about 0.001 (rounding); 0.1+ means a problem

## 7. Grade it like the judges

Five randomised runs on the practice track; the score is the median run. Then a track it has never seen.

In [ ]:
!{GRADE_PY} -m sprout_challenge.grade {ENTRY} --track practice
!{GRADE_PY} -m sprout_challenge.track --seed 7 --difficulty 0.8 --out {ROOT}/track_7.yaml
!{GRADE_PY} -m sprout_challenge.grade {ENTRY} --track {ROOT}/track_7.yaml

## 8. Watch it

In [ ]:
from IPython.display import Video
!MUJOCO_GL=egl {GRADE_PY} -m sprout_challenge.run {ENTRY} --track practice --video {ROOT}/my_run.mp4 > /dev/null
Video(str(ROOT / "my_run.mp4"), embed=True, width=720)

## 9. Download your entry

`my_entry.zip` holds `controller.py` and `policy.npz`. Unzip it into a new GitHub repository and DM the link to
[@shantanu.builds.robots](https://www.instagram.com/shantanu.builds.robots/) (see `INSTALL.md`, section 7). On
Kaggle, find it under Output on the right.

In [ ]:
shutil.make_archive(str(ROOT / "my_entry"), "zip", ENTRY)
try:
    from google.colab import files
    files.download(str(ROOT / "my_entry.zip"))
except ImportError:
    print("saved", ROOT / "my_entry.zip")

## Optional: keep checkpoints on Google Drive (Colab)

Run this **before** training (and set `CHECKPOINTS` to the Drive folder) so a disconnect doesn't lose your run.
Then set `RESUME` to the newest folder in it and train again to continue.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# CHECKPOINTS = Path("/content/drive/MyDrive/sprout_checkpoints")
# sorted(CHECKPOINTS.glob("*"))[-1:]          # the newest checkpoint, for RESUME

## Where to go from here

- **Go faster:** raise `cfg.course.speed` (and the `speed` in `controller.py`), or weight `progress` more.
- **Jump the barrier:** worth 5 s per barrier. The training world has no barriers yet: add barrier sections to
  the lanes in `training_tracks`, give the policy a reason to go over, and change the controller's steering so it
  heads at the wall instead of the open lane.
- **Robustness:** the final arena is secret. Train on more lanes (`cfg.n_lanes`), harder ones, or a different
  `cfg.track_seed`, and grade on many generated tracks.

The rules are in [RULES.md](https://github.com/shantanu-builds-robots/sprout-challenge/blob/main/RULES.md).